In [0]:
import json
import sys
from pathlib import Path

dbutils.widgets.text("repo_root", "")
supplied = dbutils.widgets.get("repo_root").strip()

if supplied:
    REPO_ROOT = Path(supplied).resolve()
else:
    REPO_ROOT = next(
        (
            candidate
            for candidate in [Path.cwd(), *Path.cwd().parents]
            if (candidate / "config/runtime.yaml").is_file()
            and (candidate / "scripts/run_config.py").is_file()
        ),
        None,
    )

if REPO_ROOT is None:
    raise ValueError(
        "Set the repo_root widget to your Workspace repository root."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.run_config import notebook_context

REPO_ROOT, config, runtime, controls = notebook_context(
    spark,
    dbutils,
)

print("Environment:", config["environment_name"])
print("Namespace:", config["namespace"])

In [0]:
from scripts.layer_runner import (
    prepare_delivery,
    run_delivery_bronze,
)

job_id = dbutils.widgets.get("job_id").strip()
job_run_id = dbutils.widgets.get("job_run_id").strip()

raw_trigger = dbutils.widgets.get("trigger_type").strip().upper()
trigger_type = (
    "MANUAL"
    if raw_trigger in {"", "MANUAL", "ONE_TIME", "RUN_JOB_TASK"}
    else "SCHEDULED"
)

delivery = prepare_delivery(
    spark,
    config,
    runtime,
    controls,
    job_id=job_id,
    job_run_id=job_run_id,
    trigger_type=trigger_type,
)

bronze_result = run_delivery_bronze(
    spark,
    config,
    controls,
    delivery,
)

if job_run_id:
    dbutils.jobs.taskValues.set(
        key="delivery_context",
        value=json.dumps(delivery),
    )

print(
    json.dumps(
        {
            "delivery": delivery,
            "bronze": bronze_result,
        },
        default=str,
        indent=2,
    )
)